In [1]:
# =========================
# 1. Install
# =========================
!pip install -q transformers datasets evaluate accelerate rouge_score

# =========================
# 2. Imports
# =========================
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
import evaluate
from sklearn.model_selection import train_test_split

# =========================
# 3. Load Final Dataset
# =========================
df = pd.read_csv("perfect_clinical_notes_summarization_dataset.csv")
df = df.dropna()

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# =========================
# 4. Load Model
# =========================
model_name = "facebook/bart-base"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

# =========================
# 5. Preprocess
# =========================
def preprocess(examples):
    model_inputs = tokenizer(
        examples["Doctor's Notes"],
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        examples["Summary"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


train_dataset = train_dataset.map(preprocess, batched=True)
val_dataset = val_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

train_dataset = train_dataset.remove_columns(["Doctor's Notes", "Summary", "__index_level_0__"])
val_dataset = val_dataset.remove_columns(["Doctor's Notes", "Summary", "__index_level_0__"])
test_dataset = test_dataset.remove_columns(["Doctor's Notes", "Summary", "__index_level_0__"])

# =========================
# 6. Dynamic Padding
# =========================
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# =========================
# 7. Training Arguments
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir="./final_clinical_model",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=4,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=5,
    fp16=False,
    logging_steps=100,
    save_total_limit=1,
)

# =========================
# 8. ROUGE Metric
# =========================
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Handle potential non-finite values and clip to valid token ID range
    if np.issubdtype(predictions.dtype, np.floating):
        predictions = np.nan_to_num(predictions, nan=tokenizer.pad_token_id, posinf=tokenizer.pad_token_id, neginf=tokenizer.pad_token_id)
    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)
    predictions = predictions.astype(np.int64)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return result

# =========================
# 9. Trainer
# =========================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# =========================
# 10. Train
# =========================
trainer.train()


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4000, training_loss=0.10899412035942077, metrics={'train_runtime': 1007.2635, 'train_samples_per_second': 15.885, 'train_steps_per_second': 3.971, 'total_flos': 1553251228139520.0, 'train_loss': 0.10899412035942077, 'epoch': 4.0})

Final Test Evaluation

In [2]:
results = trainer.evaluate(test_dataset)
print(results)


{'eval_loss': 0.08706431835889816, 'eval_rouge1': 0.5691864010418932, 'eval_rouge2': 0.4553993769581042, 'eval_rougeL': 0.5564044785027715, 'eval_rougeLsum': 0.5572988194041368, 'eval_runtime': 72.6727, 'eval_samples_per_second': 6.88, 'eval_steps_per_second': 1.72, 'epoch': 4.0}


In [3]:
for i in range(3):
    sample = df.iloc[i]["Doctor's Notes"]

    inputs = tokenizer(sample, return_tensors="pt", truncation=True).to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=5
    )

    print("\n--- Sample", i+1, "---")
    print("Generated:", tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("Actual:", df.iloc[i]["Summary"])



--- Sample 1 ---
Generated: Congestive heart failure exacerbation necessitated inpatient care. Therapeutic measures involved insulin infusion and temporary intubation and ventilation. By discharge, symptoms resolved with ongoing therapy.
Actual: Congestive heart failure exacerbation necessitated inpatient care. Therapeutic measures involved insulin infusion and temporary intubation and ventilation. By discharge, laboratory parameters normalized gradually.

--- Sample 2 ---
Generated: Acute myocardial infarction necessitated inpatient care. Therapeutic measures involved ceftriaxone and ultrasound of the abdomen. By discharge, symptoms resolved with ongoing therapy.
Actual: Hospitalization occurred due to acute myocardial infarction. Management included ceftriaxone as well as ultrasound of the abdomen. Ultimately, hemodynamics stabilized prior to discharge.

--- Sample 3 ---
Generated: Copd exacerbation necessitated inpatient care. Therapeutic measures involved furosemide and MRI brain 

In [4]:
compression = []

for i in range(200):
    original = df.iloc[i]["Doctor's Notes"]
    inputs = tokenizer(original, return_tensors="pt", truncation=True).to(model.device)
    outputs = model.generate(**inputs, max_length=128, num_beams=5)
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

    compression.append(len(summary.split()) / len(original.split()))

print("Average Compression Ratio:", sum(compression)/len(compression))
print("Estimated Reading Time Reduction:",
      (1 - (sum(compression)/len(compression))) * 100, "%")


Average Compression Ratio: 0.22811727975098686
Estimated Reading Time Reduction: 77.18827202490132 %
